In [34]:
import sys
import os
sys.path.append('..')

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json

import utilities.functions as functions
from utilities.functions import (
    retidos,
    calcula_viabilidade,
    analisar_retencao,
)

pd.set_option('display.float_format', '{:,.2f}'.format)  # 2 casas decimais
# ou


# Opcional: aumentar limite de colunas/lines
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [35]:
import os
import os
os.getcwd()
import os
import pandas as pd
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 


In [36]:

publico_janeiro_dezembro = pd.read_parquet(BASE_PATH / "gold" / "df_clientes.parquet")


In [37]:
publico_janeiro_dezembro.head()

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio
0,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,1,17,19,206.50,12.15
1,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,12,2,19,22.50,11.25
2,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,1,5,6,267.88,53.58
3,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,12,1,6,20.00,20.00
4,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,1,20,31,"1,300.89",65.04


In [38]:

df = pd.read_parquet(BASE_PATH / "gold" / "df_publico.parquet")
id_outlier=df[
    df['outlier_iqr'] & 
    df['outlier_zscore'] & 
    df['outlier_mad']]['customer_id']
len(id_outlier)
id_outlier

135    df4a3b2f7c0e259adb8e71e094405637810013013f4971...
136    df4a3b2f7c0e259adb8e71e094405637810013013f4971...
138    df4a3b2f7c0e259adb8e71e094405637810013013f4971...
252    298bdef443cf4dd3614cffa088a939fa3d3ebad163d220...
253    298bdef443cf4dd3614cffa088a939fa3d3ebad163d220...
254    298bdef443cf4dd3614cffa088a939fa3d3ebad163d220...
260    298bdef443cf4dd3614cffa088a939fa3d3ebad163d220...
359    2b44ac472c8f761db190f86ee49a7c33b3bf56792fc130...
361    2b44ac472c8f761db190f86ee49a7c33b3bf56792fc130...
557    09e4def9729099ca5ecd0e95896290d78d21a2040d102d...
559    09e4def9729099ca5ecd0e95896290d78d21a2040d102d...
567    09e4def9729099ca5ecd0e95896290d78d21a2040d102d...
569    09e4def9729099ca5ecd0e95896290d78d21a2040d102d...
580    3e4e88a5858dbd9ec7df8ccdb30a81d93c8d9152f5c94c...
581    3e4e88a5858dbd9ec7df8ccdb30a81d93c8d9152f5c94c...
635    90118aa5cd704f4f24623c77147120af73371ece404e12...
780    87de560c24c4250f17515ab5c2144b7afb0e4e0d1f971a...
Name: customer_id, dtype: objec

In [39]:
dois_pedidos=publico_janeiro_dezembro[publico_janeiro_dezembro['num_pedidos_hist']==3]
dois_pedidos

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio
19,1b669bd4b720ea0a4230ea56b8d12c675bcd4279848416...,control,1,3,3,89.44,29.81
30,fdf3cbc1ae5f2441c289b0353ef92bf7c3ef171e1130e5...,target,1,2,3,87.51,43.75
31,fdf3cbc1ae5f2441c289b0353ef92bf7c3ef171e1130e5...,target,12,1,3,51.51,51.51
92,3e4e88a5858dbd9ec7df8ccdb30a81d93c8d9152f5c94c...,target,1,2,3,284.20,142.10
93,3e4e88a5858dbd9ec7df8ccdb30a81d93c8d9152f5c94c...,target,12,1,3,192.70,192.70
120,615fa40ecaa39fd40d0c1aea7ca6d8416ef3b852da502c...,control,1,3,3,212.80,70.93


dois_pedidos=publico_janeiro_dezembro[publico_janeiro_dezembro['pedidos_sum']=='2']['customer_id'].unique()
amostra_aleatoria =publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(dois_pedidos)]
df_stats_mes = amostra_aleatoria.groupby(['order_created_month', 'pedidos_sum']).agg(
    total_clientes=('customer_id', 'nunique')
).round(2)
matriz_migracao(amostra_aleatoria,mes_0=12,mes_1=1,group_by_extra='pedidos_sum')

Calculando retencao considerando dois ou + pedidos

In [40]:
df_1,clientes_retidos_1=retidos(publico_janeiro_dezembro, mes0=12, mes1=1,pedidos=1)
print(clientes_retidos_1)

           retidos  base  taxa_retencao
is_target                              
control         30    35           0.86
target          44    47           0.94


In [41]:
retencao_pedido_1=analisar_retencao(df_1)
retencao_pedido_1

{'target_retencao': 0.93617,
 'control_retencao': 0.857143,
 'lift_absoluto': 0.079027,
 'lift_relativo_percent': 9.22,
 'target_success': 44,
 'target_total': 47,
 'control_success': 30,
 'control_total': 35,
 'z_stat': 1.192907,
 'p_value': 0.2329059191,
 'significativo': False}

Calculando retencao considerando tres ou + pedidos

In [42]:
df_2,clientes_retidos_2=retidos(publico_janeiro_dezembro, mes0=12, mes1=1,pedidos=2)
print(clientes_retidos_2)

           retidos  base  taxa_retencao
is_target                              
control         29    35           0.83
target          40    47           0.85


In [43]:
retencao_pedido_2=analisar_retencao(df_2)
retencao_pedido_2

{'target_retencao': 0.851064,
 'control_retencao': 0.828571,
 'lift_absoluto': 0.022492,
 'lift_relativo_percent': 2.71,
 'target_success': 40,
 'target_total': 47,
 'control_success': 29,
 'control_total': 35,
 'z_stat': 0.275822,
 'p_value': 0.7826845052,
 'significativo': False}

Viabilidade 

In [44]:
resultados = calcula_viabilidade(
    publico_janeiro_dezembro,
    mes_campanha=12,
    mes_seguinte=1,
    coupon_value=10.0,   
    margin_rate=0.12     
)

resultados


,mes_campanha,mes_seguinte,coupon_value,margin_rate,clientes_target_dec,clientes_control_dec,clientes_target_jan,clientes_control_jan,base_target_dec,pedidos_tot_target_dec,pedidos_tot_control_dec,pedidos_tot_target_jan,pedidos_tot_control_jan,pedidos_total_target,pedidos_total_control,pedidos_por_cliente_dec_control,pedidos_por_cliente_dec_target,pedidos_por_cliente_jan_control,pedidos_por_cliente_jan_target,inc_pedidos_dec,inc_pedidos_jan,inc_pedidos_total,gmv_tot_target_dec,gmv_tot_control_dec,gmv_tot_target_jan,gmv_tot_control_jan,gmv_por_cliente_dec_control,gmv_por_cliente_dec_target,gmv_por_cliente_jan_control,gmv_por_cliente_jan_target,inc_gmv_dec,inc_gmv_jan,inc_gmv_total,receita_ifood_t_dec,receita_ifood_c_dec,receita_ifood_t_jan,receita_ifood_c_jan,margem_incremental_dec,margem_incremental_jan,margem_incremental_total,custo_campanha,dezembro_pos,lucro_incremental,roi,gmv_total_target,gmv_total_control
0,12,1,10.00,0.12,47,35,57,43,47,210,146,439,327,649,473,4.17,4.47,7.60,7.70,13.94,5.53,19.48,"8,850.47","7,198.49","18,038.77","15,453.06",205.67,188.31,359.37,316.47,-816.07,"-2,445.52","-3,261.59","1,062.06",863.82,"2,164.65","1,854.37",198.24,310.29,508.52,470.00,592.06,38.52,0.08,"26,889.24","22,651.55"


Calculando retencao separadamente para outliers

In [45]:
publico_janeiro_dezembro.head()

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio
0,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,1,17,19,206.50,12.15
1,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,12,2,19,22.50,11.25
2,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,1,5,6,267.88,53.58
3,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,12,1,6,20.00,20.00
4,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,1,20,31,"1,300.89",65.04


In [49]:
df_outlier=publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(id_outlier)]
publico_janeiro_dezembro['is_outlier'] = publico_janeiro_dezembro['customer_id'].isin(id_outlier).astype(int)


In [50]:
publico_janeiro_dezembro.to_parquet(BASE_PATH / "gold" / "publico_janeiro_dezembro.parquet", index=False)

In [ ]:
df_2,clientes_retidos_2=retidos(df_outlier, mes0=12, mes1=1,pedidos=2)
print(clientes_retidos_2)
retencao_pedido_1=analisar_retencao(df_2)
retencao_pedido_1

           retidos  base  taxa_retencao
is_target                              
control          2     2           1.00
target           3     5           0.60


{'target_retencao': 0.6,
 'control_retencao': 1.0,
 'lift_absoluto': -0.4,
 'lift_relativo_percent': -40.0,
 'target_success': 3,
 'target_total': 5,
 'control_success': 2,
 'control_total': 2,
 'z_stat': -1.058301,
 'p_value': 0.2899184539,
 'significativo': False}

In [47]:
resultados_out = calcula_viabilidade(
    df_outlier,
    mes_campanha=12,
    mes_seguinte=1,
    coupon_value=10.0,   
    margin_rate=0.12     
)

resultados_out


,mes_campanha,mes_seguinte,coupon_value,margin_rate,clientes_target_dec,clientes_control_dec,clientes_target_jan,clientes_control_jan,base_target_dec,pedidos_tot_target_dec,pedidos_tot_control_dec,pedidos_tot_target_jan,pedidos_tot_control_jan,pedidos_total_target,pedidos_total_control,pedidos_por_cliente_dec_control,pedidos_por_cliente_dec_target,pedidos_por_cliente_jan_control,pedidos_por_cliente_jan_target,inc_pedidos_dec,inc_pedidos_jan,inc_pedidos_total,gmv_tot_target_dec,gmv_tot_control_dec,gmv_tot_target_jan,gmv_tot_control_jan,gmv_por_cliente_dec_control,gmv_por_cliente_dec_target,gmv_por_cliente_jan_control,gmv_por_cliente_jan_target,inc_gmv_dec,inc_gmv_jan,inc_gmv_total,receita_ifood_t_dec,receita_ifood_c_dec,receita_ifood_t_jan,receita_ifood_c_jan,margem_incremental_dec,margem_incremental_jan,margem_incremental_total,custo_campanha,dezembro_pos,lucro_incremental,roi,gmv_total_target,gmv_total_control
0,12,1,10.00,0.12,5,2,5,2,5,15,12,28,28,43,40,6.00,3.00,14.00,5.60,-15.00,-42.00,-57.00,"1,428.40","1,507.15","2,404.20","3,159.43",753.58,285.68,"1,579.72",480.84,"-2,339.48","-5,494.38","-7,833.85",171.41,180.86,288.50,379.13,-9.45,-90.63,-100.08,50.00,121.41,-150.08,-3.00,"3,832.60","4,666.58"
